In [1]:
!pip install autogluon.tabular[0] autogluon[tabarena]

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of opentelemetry-sdk to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.5/259.5 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is still looking at multiple versions of openxlab to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longe

In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.preprocessing import TargetEncoder
from autogluon.tabular import TabularDataset, TabularPredictor
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory
import warnings
warnings.filterwarnings('ignore')

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/playground-series-s6e2/sample_submission.csv
/kaggle/input/competitions/playground-series-s6e2/train.csv
/kaggle/input/competitions/playground-series-s6e2/test.csv


In [3]:
class config:
    SEED = 42
    N_FOLDS = 5
    TARGET = 'Heart Disease'
    
    INPUT_DIR = '/kaggle/input/playground-series-s6e2'

class_mapping = {
    'Presence': 1,
    'Absence': 0
}
rev_class_mapping = {
    0: 'Absence',
    1: 'Presence'
}

CONFIG = config()

train = pd.read_csv('/kaggle/input/competitions/playground-series-s6e2/train.csv')
train['source'] = 'train'
test = pd.read_csv('/kaggle/input/competitions/playground-series-s6e2/test.csv')
test['source'] = 'test'
sample_sub = pd.read_csv('/kaggle/input/competitions/playground-series-s6e2/sample_submission.csv')

NUMS = [col for col in train.columns if col not in ['id', 'Heart Disease', 'source']]
# NUMS = [col for col in NUMS if col not in BINS]
HIGH_CARDINALITY = [col for col in NUMS if train[col].nunique() > 40]

for df in [train, test]:
    int_cols = df.select_dtypes(include=['int64']).columns.to_list()
    float_cols = df.select_dtypes(include=['float64']).columns.to_list()
    df[int_cols] = df[int_cols].astype('int32')
    df[float_cols] = df[float_cols].astype('float32')

FEATURES = [col for col in train.columns if col not in ['id', 'Heart Disease', 'source', 'index', 'strat_feature']]
# FEATURES = [col for col in FEATURES if col not in INTER]
print(FEATURES)
print(len(FEATURES))

X = train[FEATURES]
# X = X.fillna(0)
# X_org = org[FEATURES]

y = train[CONFIG.TARGET].map(class_mapping)
# y_org = org[CONFIG.TARGET]

X_test = test[FEATURES]
# X_test = X_test.fillna(0)

['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']
13


In [4]:
BASE_FEATURES = ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol',
       'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina',
       'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']

for c in BASE_FEATURES:
    n = f'{c}_mean_te'
    TE = TargetEncoder(cv=5, random_state=42, shuffle=True)
    X[n] = TE.fit_transform(pd.DataFrame(X[c]), y).flatten()
    X_test[n] = TE.transform(pd.DataFrame(X_test[c])).flatten()

CATS = []
for col in NUMS:
    n = f'{col}_cat'
    for df in [X, X_test]:
        df[n] = df[col].astype(str).astype('category')
    CATS.append(n)

In [5]:
tabular_df = TabularDataset(pd.concat([X, y], axis=1))
label = CONFIG.TARGET
aml = TabularPredictor(label=label, eval_metric='roc_auc', problem_type='binary', path="AutogluonModelsExtremePreset", learner_kwargs={'random_state': 42})

In [6]:
aml.fit(train_data=tabular_df,
    time_limit=3600*11,
    presets='best_quality',
    num_bag_folds=5,         
    num_bag_sets=1,             
    # num_stack_levels=7,   
    refit_full=False,
    # set_best_to_refit_full=True, 
    ag_args_fit={
                # 'num_gpus': 1,
                'num_cpus': 4}, 
    # Keep random seed consistency
    # feature_generator_kwargs={'fixed_random_state': 42}
       )

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.12
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Sat Jan 17 11:20:45 UTC 2026
CPU Count:          4
Pytorch Version:    2.9.0+cpu
CUDA Version:       CUDA is not available
Memory Avail:       29.71 GB / 31.35 GB (94.8%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=5, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` val

In [7]:
leaderboard = aml.leaderboard()
leaderboard.to_csv('leaderboard.csv', index=False)

In [8]:
leaderboard

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,0.955740,roc_auc,56.652906,12750.839332,0.121463,77.705095,2,True,29
1,CatBoost_r137_BAG_L1,0.955703,roc_auc,1.097164,1536.156649,1.097164,1536.156649,1,True,21
2,CatBoost_BAG_L1,0.955695,roc_auc,1.550374,1607.112281,1.550374,1607.112281,1,True,5
3,CatBoost_r177_BAG_L1,0.955693,roc_auc,0.643443,718.284884,0.643443,718.284884,1,True,12
4,CatBoost_r13_BAG_L1,0.955665,roc_auc,2.041254,2687.045655,2.041254,2687.045655,1,True,23
5,LightGBM_r96_BAG_L1,0.955534,roc_auc,21.963320,285.877738,21.963320,285.877738,1,True,17
6,XGBoost_r89_BAG_L1,0.955459,roc_auc,2.645634,455.675879,2.645634,455.675879,1,True,27
7,CatBoost_r9_BAG_L1,0.955434,roc_auc,2.671258,952.546844,2.671258,952.546844,1,True,16
8,XGBoost_BAG_L1,0.955336,roc_auc,2.651852,419.806391,2.651852,419.806391,1,True,9
9,LightGBMXT_BAG_L1,0.955306,roc_auc,4.006611,93.078835,4.006611,93.078835,1,True,1


In [9]:
model_names = aml.model_names(can_infer=True)
oof_dict = {}
test_dict = {}
for model_name in model_names:
    if not model_name.endswith('_FULL'):
        try:
            oof_probs = aml.predict_proba_oof(model=model_name).iloc[:, 1]
            oof_dict[model_name] = oof_probs.values

            test_probs = aml.predict_proba(X_test, model=model_name).iloc[:, 1]
            test_dict[model_name] = test_probs.values

            print(f'{model_name} done ...')

        except Exception as e:
            print(f'Could not extract {model_name}: {e}')

LightGBMXT_BAG_L1 done ...
LightGBM_BAG_L1 done ...
RandomForestGini_BAG_L1 done ...
RandomForestEntr_BAG_L1 done ...
CatBoost_BAG_L1 done ...
ExtraTreesGini_BAG_L1 done ...
ExtraTreesEntr_BAG_L1 done ...
NeuralNetFastAI_BAG_L1 done ...
XGBoost_BAG_L1 done ...
NeuralNetTorch_BAG_L1 done ...
LightGBMLarge_BAG_L1 done ...
CatBoost_r177_BAG_L1 done ...
NeuralNetTorch_r79_BAG_L1 done ...
LightGBM_r131_BAG_L1 done ...
NeuralNetFastAI_r191_BAG_L1 done ...
CatBoost_r9_BAG_L1 done ...
LightGBM_r96_BAG_L1 done ...
NeuralNetTorch_r22_BAG_L1 done ...
XGBoost_r33_BAG_L1 done ...
ExtraTrees_r42_BAG_L1 done ...
CatBoost_r137_BAG_L1 done ...
NeuralNetFastAI_r102_BAG_L1 done ...
CatBoost_r13_BAG_L1 done ...
RandomForest_r195_BAG_L1 done ...
LightGBM_r188_BAG_L1 done ...
NeuralNetFastAI_r145_BAG_L1 done ...
XGBoost_r89_BAG_L1 done ...
LightGBM_r130_BAG_L1 done ...
WeightedEnsemble_L2 done ...


In [10]:
all_model_oofs = pd.DataFrame(oof_dict)
all_model_test = pd.DataFrame(test_dict)

all_model_oofs.to_csv('autogluon_all_oofs.csv', index=False)
all_model_test.to_csv('autogluon_all_test.csv', index=False)

print("\n--- Process Complete ---")
print(f"OOF Shape: {all_model_oofs.shape}")
print(f"Test Shape: {all_model_test.shape}")


--- Process Complete ---
OOF Shape: (630000, 29)
Test Shape: (270000, 29)
